# 09 — Walk-Forward Backtest

Validates the Markov model via walk-forward testing on historical prices:

1. **Sliding window** — regime estimation on a rolling history
2. **Forecast** — P(up) and CI at each step
3. **Realised outcome** — did price land in CI? Direction correct?
4. **Metrics** — Brier score, directional accuracy, CI coverage
5. **Bootstrap CIs** — confidence intervals for accuracy metrics

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.data.prices import fetch_historical_prices
from research.backtest.walk_forward import walk_forward
from research.backtest.metrics import (
    brier_score,
    directional_accuracy,
    ci_coverage,
    mean_absolute_error,
    bootstrap_directional_ci,
    bootstrap_brier_ci,
    calibration_table,
)

## 1. Load Data

Fetch enough history for a meaningful backtest.

In [ ]:
TICKER = 'BTC'
DAYS = 730
WARMUP = 120
HORIZON = 7
STRIDE = 10

prices_df = fetch_historical_prices(TICKER, days=DAYS)
prices = prices_df['close'].tolist()
dates = prices_df['date'].tolist()

print(f"Prices: {len(prices)} days")
print(f"Warmup: {WARMUP}  Horizon: {HORIZON}  Stride: {STRIDE}")
print(f"Expected steps: ~{(len(prices) - WARMUP - HORIZON) // STRIDE}")

## 2. Run Walk-Forward Backtest

In [ ]:
result = walk_forward(
    prices=prices,
    horizon=HORIZON,
    warmup=WARMUP,
    stride=STRIDE,
    decay_rate=0.97,
)

steps = result.steps
print(f"Steps: {len(steps)}")
print(f"Errors: {len(result.errors)}")
if result.errors:
    for e in result.errors[:3]:
        print(f"  - {e}")

dir_ci = bootstrap_directional_ci(steps, n_bootstrap=1000)
brier_ci = bootstrap_brier_ci(steps, n_bootstrap=1000)

print(f"Directional accuracy: {direction*100:.1f}%")
print(f"  95% CI: [{dir_ci['lower']*100:.1f}%, {dir_ci['upper']*100:.1f}%]")
print(f"  Bootstrap mean: {dir_ci['mean']*100:.1f}%")
print()
print(f"Brier score: {brier:.4f}")
print(f"  95% CI: [{brier_ci['lower']:.4f}, {brier_ci['upper']:.4f}]")
print(f"  Bootstrap mean: {brier_ci['mean']:.4f}")

# Note: with limited steps the bootstrap CI can be wide; this is informational
if dir_ci['lower'] > 0.40:
    print('Bootstrap gate: PASSED')
else:
    print(f"Bootstrap lower bound {dir_ci['lower']:.1%} — consider more data or longer horizon")

In [ ]:
# No NaN
for s in steps:
    assert not np.isnan(s.predicted_prob)
    assert not np.isnan(s.predicted_return)
    assert not np.isnan(s.ci_lower)
    assert not np.isnan(s.ci_upper)

# Probabilities in [0, 1]
for s in steps:
    assert 0 <= s.predicted_prob <= 1

# CI lower < upper
for s in steps:
    assert s.ci_lower < s.ci_upper

print('Tier 1 robustness: PASSED')

## 4. Tier 2 — Regression Gates

Aggregate metrics that catch model degradation.

In [ ]:
brier = brier_score(steps)
direction = directional_accuracy(steps)
coverage = ci_coverage(steps)
mae = mean_absolute_error(steps)

print(f"Brier score:           {brier:.4f}  (lower = better calibration)")
print(f"Directional accuracy:  {direction*100:.1f}%")
print(f"CI coverage:           {coverage*100:.1f}%  (target ~68-95%)")
print(f"Mean absolute error:   {mae*100:.2f}%")

# Thresholds from TypeScript tests
assert brier < 0.42, f'Brier score {brier:.3f} exceeds regression threshold'
print('Tier 2 regression gates: PASSED')

## 5. Bootstrap Confidence Intervals

Bootstrap the directional accuracy and Brier score to get uncertainty around the metrics.

In [ ]:
dir_ci = bootstrap_directional_ci(steps, n_bootstrap=1000)
brier_ci = bootstrap_brier_ci(steps, n_bootstrap=1000)

print(f"Directional accuracy: {direction*100:.1f}%")
print(f"  95% CI: [{dir_ci['lower']*100:.1f}%, {dir_ci['upper']*100:.1f}%]")
print(f"  Bootstrap mean: {dir_ci['mean']*100:.1f}%")
print()
print(f"Brier score: {brier:.4f}")
print(f"  95% CI: [{brier_ci['lower']:.4f}, {brier_ci['upper']:.4f}]")
print(f"  Bootstrap mean: {brier_ci['mean']:.4f}")

assert dir_ci['lower'] > 0.40, f"Directional accuracy lower bound {dir_ci['lower']:.1%} too low"
print('Bootstrap gate: PASSED')

## 6. Visualise Step-by-Step

Plot realised returns vs predicted probabilities and CI coverage.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Predicted probability vs realised return
ax = axes[0, 0]
x = [s.predicted_prob for s in steps]
y = [s.realised_return * 100 for s in steps]
colors = ['green' if s.direction_correct else 'red' for s in steps]
ax.scatter(x, y, c=colors, alpha=0.6, s=60)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Predicted P(up)')
ax.set_ylabel('Realised Return (%)')
ax.set_title('Predicted Probability vs Realised Return')
ax.grid(True, alpha=0.3)

# Directional accuracy over time
ax = axes[0, 1]
window = 10
rolling_acc = [
    np.mean([s.direction_correct for s in steps[max(0, i-window):i+1]]) * 100
    for i in range(len(steps))
]
ax.plot(rolling_acc, linewidth=2, color='steelblue')
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.axhline(60, color='green', linestyle=':', alpha=0.5, label='60%')
ax.set_xlabel('Step')
ax.set_ylabel('Rolling Accuracy (%)')
ax.set_title(f'Rolling Directional Accuracy (window={window})')
ax.legend()
ax.grid(True, alpha=0.3)

# CI coverage over time
ax = axes[1, 0]
rolling_cov = [
    np.mean([s.in_ci for s in steps[max(0, i-window):i+1]]) * 100
    for i in range(len(steps))
]
ax.plot(rolling_cov, linewidth=2, color='purple')
ax.axhline(95, color='green', linestyle=':', alpha=0.5, label='95% target')
ax.axhline(68, color='blue', linestyle=':', alpha=0.5, label='68% target')
ax.set_xlabel('Step')
ax.set_ylabel('Rolling CI Coverage (%)')
ax.set_title(f'Rolling CI Coverage (window={window})')
ax.legend()
ax.grid(True, alpha=0.3)

# Calibration plot
ax = axes[1, 1]
calib = calibration_table(steps, bins=5)
if calib:
    bin_mids = [c['bin_mid'] for c in calib]
    observed = [c['observed_freq'] for c in calib]
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.scatter(bin_mids, observed, s=100, color='steelblue', zorder=3)
    for c in calib:
        ax.annotate(f"n={c['count']}", (c['bin_mid'], c['observed_freq']),
                   textcoords='offset points', xytext=(0, 10), ha='center', fontsize=8)
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Observed Frequency')
    ax.set_title('Reliability Diagram')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No calibration data', ha='center', va='center')
    ax.set_title('Reliability Diagram')

plt.tight_layout()
plt.show()

## 7. Sensitivity: Horizon & Decay

How does directional accuracy change with horizon and decay rate?

In [ ]:
horizons = [7, 14, 21, 30]
decay_rates = [0.92, 0.95, 0.97, 0.99]

sens_results = []
for horizon in horizons:
    for decay in decay_rates:
        res = walk_forward(prices, horizon=horizon, warmup=WARMUP, stride=STRIDE, decay_rate=decay)
        if res.steps:
            acc = directional_accuracy(res.steps)
            brier = brier_score(res.steps)
            sens_results.append({
                'horizon': horizon,
                'decay': decay,
                'accuracy': acc,
                'brier': brier,
                'steps': len(res.steps),
            })

sens_df = pd.DataFrame(sens_results)
pivot = sens_df.pivot(index='horizon', columns='decay', values='accuracy')

fig, ax = plt.subplots(figsize=(10, 5))
for decay in decay_rates:
    subset = sens_df[sens_df['decay'] == decay]
    ax.plot(subset['horizon'], subset['accuracy'] * 100, marker='o', linewidth=2, label=f'decay={decay}')
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_xlabel('Horizon (days)')
ax.set_ylabel('Directional Accuracy (%)')
ax.set_title(f'{TICKER}: Accuracy vs Horizon & Decay')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(sens_df[['horizon', 'decay', 'accuracy', 'steps']].to_string(index=False))

## 8. Step Table

First and last few steps for manual inspection.

In [ ]:
step_df = pd.DataFrame([
    {
        'start_idx': s.start_idx,
        'pred_prob': s.predicted_prob,
        'pred_return': s.predicted_return,
        'ci_low': s.ci_lower,
        'ci_high': s.ci_upper,
        'real_return': s.realised_return,
        'direction_ok': s.direction_correct,
        'in_ci': s.in_ci,
    }
    for s in steps
])
print(step_df.head(10).to_string(index=False))
print('---')
print(step_df.tail(5).to_string(index=False))